# 多通道逻辑分析仪与协议分析 — 上板演示

PYNQ-Z2（XC7Z020）· PL 侧全部自主实现（Verilog）· PS 侧只做搬运与显示。

| 项 | 值 |
|---|---|
| 通道数 | 8 |
| 物理缓冲 | 8192 点（BRAM 环形，读写指针都在硬件里） |
| 采样率 | 62.5 MHz / DIV（最高 62.5 MSa/s） |
| 触发 | 上升沿 / 下降沿 / 高电平 / 低电平 / **协议字节匹配**，含预触发 |
| 时序统计 | 脉宽·周期 256 桶直方图 + 硬件分位数扫描 |
| 板内自测源 | 8 路方波（`CTRL.bit2`）；I2C 协议源（`CTRL.bit3`） |

**本 notebook 全部单元格都不需要外部接线**：
PL 内部有 8 路频率依次倍增的方波（接错通道从波形上一眼看出），
还有一个 I2C 协议源循环发 `A0 / 55 / AA`，用来验证解码与协议触发。

> 图里的标签用英文：板子上唯一的 CJK 字体（Droid Sans Fallback）**不含拉丁字母**，
> 而 matplotlib 3.5 不支持跨字体逐字形回退，中英混排会大面积报缺字。

## 0. 初始化

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from la_driver import LogicAnalyzer

la = LogicAnalyzer()          # 加载 la_sys.bit
print(la)
print('IP  :', la.ip)
print('基地址 :', hex(la.base))

## 1. 采集一次

`capture()` 一步走完契约里的流程：配置 → 软复位 → 布防 → 等抓满 → 按时间顺序读回。

触发通道选 CH5（`cnt[21]`，周期约 67 ms），窗口 65 ms，正好能看一个完整周期。

In [ ]:
samples, meta = la.capture(div=1000, pre_trig=2000, trig_chan=5,
                           read_depth=4096)

print('采样率      : %.1f kHz  (DIV=%d)' % (meta['sample_rate']/1e3, meta['div']))
print('等 FULL 用时 : %.0f ms' % (meta['t_wait']*1e3))
print('读回 4096 点 : %.0f ms' % (meta['t_read']*1e3))
print('STOP_PTR    : 0x%X   (时间顺序上最老那个点的物理地址)' % meta['stop_ptr'])
print('非零点      : %d / %d' % (int(np.count_nonzero(samples)), len(samples)))

In [ ]:
la.plot(samples, chans=[0,1,2,3,4,5,6,7], meta=meta,
        title='Built-in test source: 8 channels, octave-spaced')
t_trig = meta['pre_trig'] * meta['div'] / la.CLK_HZ * 1e6
plt.axvline(t_trig, color='r', ls='--', lw=1, label='trigger point')
plt.legend(loc='upper right')
plt.tight_layout(); plt.show()

红色虚线是触发点。它左边是**触发之前**就已经存在缓冲里的数据——
这就是预触发：触发不是"开始记录"，而是"让一直在循环的记录停下来"。

## 2. 触发：从"抓到了"到"抓到想看的那一段"

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

for ax, mode, name in ((axes[0], la.TRIG_RISING,  'rising edge'),
                       (axes[1], la.TRIG_FALLING, 'falling edge')):
    s, m = la.capture(div=1500, pre_trig=4000, trig_chan=5,
                      trig_mode=mode, read_depth=8192)
    a = la.plot(s, chans=[5], meta=m, ax=ax, title='CH5 trigger: %s' % name)
    a.axvline(m['pre_trig'] * m['div'] / la.CLK_HZ * 1e6,
              color='r', ls='--', lw=1)

plt.tight_layout(); plt.show()

同一个信号，换个触发条件，抓到的就是完全不同的那一段。
**这件事必须在 PL 里做**：PS 从"读到状态"到"发出停止命令"隔着 Linux 调度，
微秒级起步；而线上一个时钟沿是 16 ns，等它反应过来那个沿早就过去了。

## 3. 协议解码（板内自测，不用接线）

PL 里有 UART / SPI / I2C 三个解码器**并行**工作，`PROTO_SEL` 选哪个接到匹配器。
三个解码器各自盯**固定**的通道，不能改：

| 协议 | 通道 |
|---|---|
| UART | CH0 = RX |
| SPI | CH1 = SCLK，CH2 = MOSI，CH3 = CS_N |
| I2C | **CH4 = SCL，CH5 = SDA** |

`CTRL.bit3 = 1` 时，板内自测源的 **CH4/CH5 会换成 I2C 源的 SCL/SDA**
（源循环发 `START + A0 + 55 + AA + STOP`，应答位由它自己模拟），
于是**不接任何外线也能验证整条解码通路**。下面这一格做的就是这件事。

In [ ]:
s, m = la.capture(div=20, pre_trig=2000, trig_chan=4,
                  trig_mode=la.TRIG_RISING, read_depth=4096,
                  proto_sel=la.PROTO_I2C)

print('采样率        : %.1f kHz (DIV=%d)' % (m['sample_rate'] / 1e3, m['div']))
print('解出的字节    : 0x%02X   （源发的三个是 A0 / 55 / AA）' % la.proto_byte())
print('解码器状态    :', la.proto_status())
print('CH4(SCL) 边沿 : %d 次' % len(la.edges(s, 4)))

la.plot(s, chans=[4, 5], meta=m,
        title='Built-in I2C source: SCL(CH4) / SDA(CH5)')
plt.tight_layout(); plt.show()

## 4. 协议内容触发（板内自测，不用接线）

`TRIG_MODE = 4` 时触发源换成协议匹配脉冲：**等总线上出现指定字节才开始抓**。
这是逻辑分析仪最实用的模式——总线上跑的帧成千上万，只想看其中某一条命令。

板内 I2C 源会周期性地发出 `0xAA`，所以下面这格能真的触发，不用外接信号。

In [ ]:
s, m = la.capture(div=20, pre_trig=1000, read_depth=2048,
                  trig_mode=la.TRIG_PROTO, trig_value=0xAA,
                  proto_sel=la.PROTO_I2C, timeout=3.0)

print('触发模式 : PROTO_SEL=I2C，等总线上出现字节 0xAA')
print('抓满用时 : %.0f ms' % (m['t_wait'] * 1e3))

la.plot(s, chans=[4, 5], meta=m,
        title='Protocol-content trigger: captured after 0xAA on the bus')
plt.tight_layout(); plt.show()

## 5. 时序统计（硬件直方图 + 分位数）

脉宽/周期在采集通路上顺手累加成 256 桶直方图；分位数只需扫 256 个桶。
**计算量从 O(样本数) 降到 O(桶数)。**

In [ ]:
la.stats_clear()                    # bit1 是保持型：置1 -> 等 clr_busy 落 -> 置0
la.stats_enable(chan=7)             # 统计通道 = TRIG_CHAN
time.sleep(1.0)

t = la.stats_scan()                 # 顺序要紧：必须先扫描，分位数才算得出来
st = la.stats_read()

print('样本数 : 宽度 %d / 周期 %d' % (st['n_width'], st['n_period']))
print('饱和 %d 次，丢弃 %d 次' % (st['n_sat'], st['n_drop']))
print('硬件分位数扫描耗时 : %s' % ('%.1f us' % (t * 1e6) if t else '超时'))
print('宽度 P50/P95/P99 (clk) :', st['w_ps'])
print('周期 P50/P95/P99 (clk) :', st['p_ps'])
print()
print('单位是采样时钟：1 clk = %.1f ns @ %.1f MHz' % (1e9 / la.CLK_HZ, la.CLK_HZ / 1e6))

In [ ]:
hist = la.histogram('w')
plt.figure(figsize=(12, 3))
plt.bar(np.arange(256), hist, width=1.0)
plt.xlabel('bucket  (16 clk each)'); plt.ylabel('count')
plt.title('Pulse-width histogram - hardware, 256 bins')
plt.tight_layout(); plt.show()

la.stats_enable(on=False)

## 6. 说明

- 所有波形、触发、协议解码、直方图都在 **PL 里完成**；PS 只负责搬运和画图。
- 采样率 = `62.5 MHz / DIV`，DIV 最小 1，即上限 62.5 MSa/s。
  这个频率是实测出来的（`la_pynq/bringup/measure_rate.py`），不是照抄板卡手册。
- 未实现项（报告里如实写明）：`TRIG_MASK` 掩码比较、`CHAN_EN` 通道使能。
  物理缓冲深度是 RTL 参数 8192，`0x0C DEPTH` 寄存器只控制"这次读回多少点"。